# Audio Evaluation — WER, MOS, UTMOS, MMAU, FAD, and the Open Leaderboards Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: WER with normalization

In [ ]:
```python

from jiwer import wer, Compose, ToLowerCase, RemovePunctuation, Strip

transform = Compose([ToLowerCase(), RemovePunctuation(), Strip()])

score = wer(

    truth="Please turn on the lights.",

    hypothesis="please turn on the light",

    truth_transform=transform,

    hypothesis_transform=transform,

)

# ~0.17

In [ ]:
```

### Step 2: TTS round-trip WER

In [ ]:
```python

def ttr_wer(tts_model, asr_model, texts):

    errors = []

    for txt in texts:

        audio = tts_model.synthesize(txt)

        recog = asr_model.transcribe(audio)

        errors.append(wer(truth=txt, hypothesis=recog))

    return sum(errors) / len(errors)

In [ ]:
```

### Step 3: SECS for voice cloning

In [ ]:
```python

from speechbrain.inference.speaker import EncoderClassifier

sv = EncoderClassifier.from_hparams("speechbrain/spkrec-ecapa-voxceleb")

emb_ref = sv.encode_batch(load_wav("reference.wav"))

emb_clone = sv.encode_batch(load_wav("cloned.wav"))

secs = torch.nn.functional.cosine_similarity(emb_ref, emb_clone, dim=-1).item()

In [ ]:
```

### Step 4: FAD for music generation

In [ ]:
```python

from frechet_audio_distance import FrechetAudioDistance

fad = FrechetAudioDistance()

score = fad.get_fad_score("generated_folder/", "reference_folder/")

In [ ]:
```

### Step 5: EER for speaker verification (same code as Lesson 6)

In [ ]:
```python

def eer(same_scores, diff_scores):

    thresholds = sorted(set(same_scores + diff_scores))

    best = (1.0, 0.0)

    for t in thresholds:

        far = sum(1 for s in diff_scores if s >= t) / len(diff_scores)

        frr = sum(1 for s in same_scores if s < t) / len(same_scores)

        if abs(far - frr) < best[0]:

            best = (abs(far - frr), (far + frr) / 2)

    return best[1]

In [ ]:
```

## Exercises

In [ ]:
1. **Easy.** Run `code/main.py`. Compute WER / CER / EER / SECS / FAD-ish / MMAU-ish on toy inputs.
2. **Medium.** Build a TTS round-trip WER harness. Run your Kokoro or F5-TTS output through Whisper. Compute WER over 50 prompts. Flag prompts with WER &gt; 10%.
3. **Hard.** Score your Lesson 10 LALM choice on MMAU-Pro speech + multi-audio subsets (50 items each). Report per-category accuracy and compare with the published number.